# SLAC XSIF to Bmad conversion

In [1]:
import subprocess
import sys
import re
import os

# Remove Comment blocks

In [2]:
def remove_comment_blocks(lines, prefix='!=!'):
    """
    Removes comment blocks of the form:
    COMMENT
    text
    ...
    ENDCOMMENT
    
    """
    out = []
    inside = False
    for line in lines:
        if line.strip().upper().startswith('COMMENT'):
            inside = True
            out.append(prefix+line)
        if line.strip().upper().startswith('ENDCOMMENT'):
            assert inside, 'ERROR: nexted comments not supported.'
            inside = False
            out.append(prefix+line)
        elif inside:
            out.append(prefix+line)
        else:
            out.append(line)
    return out

        
        
        

# Change Set commands

In [3]:
re.sub(r' *SET *, * (\w*) *, *(\w*)', r'\1 = \2', 'SET, key, val')

'key = val'

In [4]:
def replace_set(line):
    return re.sub(r' *SET *, * (\w*) *, *(\w*)', r'\1 = \2', line)

#def replace_set2(line):
#    return re.sub(r' (\w*), * (\w*) *= *(\w*)', r'\1 = \2', line)
    
replace_set('SET,  afa, afa, 1a')

#replace_set2('QUM1,   K1=')

'afa = afa, 1a'

In [5]:
def replace_set_commands(lines):
    return [replace_set(line) for line in lines]

# Expand names, correct matrix element syntax

In [6]:
def fix_matrix(line):
    """
    Replaces RM(1,2) with R12
    """
    return re.sub(r'RM\(([1-6]),([1-6])\)', r'R\1\2', line)
fix_matrix('RM(3,4)')    

'R34'

In [7]:
FULLNAMES = {
    'APER':'aperture',
    'LCAV':'lcavity',
    'IMON':'MARKER',
    'WIRE':'MARKER',
    'PROF':'MONITOR',
    'BLMO':'MONITOR'
}
def expand_names(line):
    for k, v in FULLNAMES.items():
        line = re.sub(k,v,line)
    return line





expand_names('  afa APER BLMO')

'  afa aperture MONITOR'

In [8]:
def fix_names(lines):
    out = []
    for line in lines:
        line = expand_names(line)
        line = fix_matrix(line)
        out.append(line)
    return out
fix_names(['sfafasfa safa APER RM(1,2)'])

['sfafasfa safa aperture R12']

# Folding and unfolding comments

In [9]:
c = '!'
c2 = '! INLINE--#'
c3 = '! SIMPLE --#'
empty = '! EMPTY --#'

In [10]:
def unfold_comments(lines):
    '''
    Break a line with a comment at the end into two pieces. Preserve whitespace.
    '''
    newlines = []
    for line in lines:
        ix = line.find(c)
        if ix>-1:
            # There is a comment in this line
            # Get whitespace too
            m=re.search('\s*'+c+'.*', line)
            firstpart = line[0:ix].rstrip()
            if firstpart.strip() =='':
                # Simple comment with whitespace
                newlines.append(c3+m.group(0))
            else:
                # Inline comment
                newlines.append(c2+m.group(0))
                newlines.append(line[0:ix].rstrip())
        elif len(line.strip()) ==0:
            # Empty line
            newlines.append(empty)
        else:
            newlines.append(line.rstrip())

    return newlines



In [11]:
def fold_comments(lines):
    '''
    Fold lines starting with c2 into next line
    '''
    newlines = []
    x = None
    for line in lines:
        line = line.rstrip()
        ix = line.find(c2)
        ix3 = line.find(c3)
        if line.strip() == empty :
            # Empty line
            newlines.append('')
        elif ix3 ==0:
            # Simple comment
            newlines.append(line[len(c3):])
        elif ix == 0:
            # No comments, just return line
            x = line[len(c2):]
        else:
            if x:
                newlines.append(line[0:] + x )
                x = None
            else:
                newlines.append(line.rstrip())
    return newlines

In [12]:
def test():
    L0 = ['123\n', '123    !comment\n', '   \n', '  !simple comment\n', '123!456#789\n']
    L1 = unfold_comments(L0)
    L2 = fold_comments(L1)
    print(L0)
    print(L1)
    print(L2)
test()

['123\n', '123    !comment\n', '   \n', '  !simple comment\n', '123!456#789\n']
['123', '! INLINE--#    !comment', '123', '! EMPTY --#', '! SIMPLE --#  !simple comment', '! INLINE--#!456#789', '123']
['123', '123    !comment', '', '  !simple comment', '123!456#789']


# AML Translate

In [13]:
TRANSLATE_BIN='/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver'

In [14]:
assert os.path.exists(TRANSLATE_BIN)

In [15]:
cmd = [TRANSLATE_BIN, '-bmad', 'UND.xsif']
#p = subprocess.run(cmd, capture_output=True)
#!/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver -bmad LTU.xsif

In [16]:
def translate_xsif_to_bmad(infile, TRANSLATE_BIN='/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver', verbose=True):
    cmd = [TRANSLATE_BIN, '-bmad', infile] 
    p = subprocess.run(cmd, capture_output=True)
    if verbose:
        print('Translating:', cmd)
        print(p)
    if 'ERROR' in str(p.stdout):
        print('Errors in translation')
    

# Desplitting (in Bmad)

In [17]:
def desplit_ele(line, double_length=True):
    """
    De-splits elements. Converts form:
    ele_full: line (ele, other...eles, ele) to:
    ele_full: line = (ele)
    ele[L] = 2*ele[L]
    other eles[superimpose] = T
    other eles[ref] = ele
    
    Example:
    
    !Original split line: line = (qsx16, xcsx16, ycsx16, qsx16)
    qsx16_full: line = (qsx16)
    qsx16[L] = 2*qsx16[L]
    xcsx16[superimpose] = T
    xcsx16[ref] = qsx16
    ycsx16[superimpose] = T
    ycsx16[ref] = qsx16
    
    """
    # Check if this is a _full line
    original_line = line
    
    s = line.split(':')
    if len(s) ==1:
        return line
    ix = s[0].find('_full')
    if ix <0:
        return line
    # Should be. 
    ele, line = line.split(':')
    name = ele.split('_full')[0].lower()
    eles = [e.strip().lower() for e in (line.split('(')[1].split(')')[0]).split(',')]
    # Make sure this is true
    if eles[0] != name or eles[-1] != name:
        print('Warning: different starting and ending ele names: '+ original_line, '\n   Skipping.')
        return original_line
    
    #assert eles[0] == name
    #assert eles[-1] == name
    insideeles = eles[1:-1]
    
    lines = ['\n', '!Old split line:'+line]
    lines.append(name+'_full: line = ('+name+')')
    if double_length:
        lines.append(name+'[L] = 2*'+name+'[L]')
    for e in insideeles:
        lines.append(e+'[superimpose] = T')
        lines.append(e+'[ref] = '+name)
    lines.append('\n')
    return '\n'.join(lines)
        
def desplit_eles(lines):
    return [desplit_ele(line) for line in lines]
    
line0 = 'qsx16_full: line = (qsx16, xcsx16, ycsx16, qsx16)'    
line1 = 'qsx16_full: line = (qsx16,  qsx16a)'  
print(desplit_ele(line1))

   Skipping.
qsx16_full: line = (qsx16,  qsx16a)


In [18]:
desplit_eles(['fafaa', line0])

['fafaa',
 '\n\n!Old split line: line = (qsx16, xcsx16, ycsx16, qsx16)\nqsx16_full: line = (qsx16)\nqsx16[L] = 2*qsx16[L]\nxcsx16[superimpose] = T\nxcsx16[ref] = qsx16\nycsx16[superimpose] = T\nycsx16[ref] = qsx16\n\n']

# Custom element replacements (in Bmad)

In [19]:
def replace_element(lines, ele_name, new_ele, verbose=True):
    """
    Searches through lines for:
    ele_name: <some definition>
    , comments it out, and writes new_ele string below the comment. 
    Considers & to continue the line. 
    """
    newlines = []
    inele = False
    for line in lines:
        if inele:
            # Continued element definition.
            newlines.append('!old '+line)
            if line.strip()[-1] != '&':
                inele=False
            continue
        
        s = line.split(':')
        if len(s) == 1:
            newlines.append(line)
            continue
        if s[0].strip().lower() != ele_name:
            newlines.append(line)
            continue
        # Should be a match    
        if verbose:
            print('Found ele:', ele_name)
        newlines.append(new_ele)
        newlines.append('!old: '+line)
        if line.strip()[-1] == '&':
            # Element definition continues
            inele = True
    return newlines
#for x in replace_element(open('UND.bmad'), 'pshx', 'DUMMY: wiggler'):
#    print(x)
    

In [20]:
NEWELES = {}



NEWELES['umasx'] = """
!------- SXR Undulator -------
my_umasx_k = 5.0
umasx: wiggler, 
        type = "UMASX",
        L_period = 0.039, 
        n_period = 87, 
        b_max = my_umasx_k * 2*pi*m_electron / (c_light * 0.039), 
        L = 87*0.039, 
        ds_step = 0.039*10
        
umasx[L] = umasx[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""

NEWELES['umahx'] = """
!------- HXR Undulator -------
my_umahx_k = 2.0
umahx: wiggler, 
        type = "UMAHX",
        L_period = 0.026, 
        n_period = 129, 
        b_max = my_umahx_k * 2*pi*m_electron / (c_light * 0.026), 
        L = 129*0.026, 
        tilt=pi/2,
        ds_step = 0.026*10
        
umahx[L] = umahx[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
    """




NEWELES['pssx'] = """
!------- SXR Phase Shifter -------
!
! B_max = 2pi/lambda * sqrt(2*PHASE_INTEGRAL / L)
! 
pssx_phase_integral = 3814e-9  !T^2 m^3, maximum, from: T^2mm^3 (180-3814)
pssx_L        = 0.0825   ! m 
pssx_L_period = 0.075 ! m 
pssx: wiggler, type = "PSSX", 
    L = pssx_L,
    b_max = 2*pi / pssx_L_period * sqrt(2 * pssx_phase_integral / pssx_L  ),
    n_period = 1
pssx[L] = pssx[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""



NEWELES['pshx'] = """
!------- HXR Phase Shifter -------
!
! B_max = 2pi/lambda * sqrt(2*PHASE_INTEGRAL / L)
! 
pshx_phase_integral = 490e-9  !T^2 m^3, maximum, from: T^2mm^3 (80-490)
pshx_L        = 0.0495 ! m 
pshx_L_period = 0.045 ! m 
pshx: wiggler, type = "PSHX", 
    L = pshx_L,
    b_max = 2*pi / pshx_L_period * sqrt(2 * pshx_phase_integral / pshx_L  ),
    n_period = 1
pshx[L] = pshx[L]/2 ! Will be doubled in desplitting process. 
!---------------------------------
"""

In [21]:
def replace_eles(lines, replacements):
    """
    
    """
    newlines = lines
    for k in replacements:
        newlines = replace_element(newlines, k, replacements[k])
    return newlines

# Full conversion

In [22]:
def prepare_xsif(xsif_file, save=True):
    """
    Prepares an XSIF file for conversion. 
    """
    path, file = os.path.split(xsif_file)
    basename = file.split('.')[0]
    outname = basename+'.bmad'
    print('Preparing', file)
    
    with open(xsif_file) as f:
        lines = f.readlines()
        
    # Remove comment blocks
    lines = remove_comment_blocks(lines)
    
    # Replace set commands
    lines = replace_set_commands(lines)
    
    # Fix names, matrix
    lines = fix_names(lines)
    
    # Unfold comments
    lines = unfold_comments(lines)
    
    # Save
    if save:
        os.rename(xsif_file, xsif_file+'_save')
    
    
    with open(xsif_file, 'w') as f:
        for line in lines:
            f.write(line+'\n')
            
#prepare_xsif('LTU.xsif')

In [33]:
translate_xsif_to_bmad('LTU.xsif')

Translating: ['/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver', '-bmad', 'LTU.xsif']
CompletedProcess(args=['/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver', '-bmad', 'LTU.xsif'], returncode=-6, stdout=b'', stderr=b'dyld: Library not loaded: /Users/chrisonian/Code/bmad_svn/accelerator-ml-code/xerces-c-3.1.4/local/lib/libxerces-c-3.1.dylib\n  Referenced from: /Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver\n  Reason: image not found\n')


In [30]:
with open('LTU.bmad') as f:
    lines = f.readlines()

FileNotFoundError: [Errno 2] No such file or directory: 'LTU.bmad'

In [ ]:
lines2 = replace_eles(lines, NEWELES)

In [ ]:
NEWELES.keys()

In [23]:
def finalize_bmad(bmad_file, replacements={}):
    with open(bmad_file) as f:
        lines = f.readlines()
   
    # Custom element replacements
    lines = replace_eles(lines, replacements)
        
    # Desplit
    lines = desplit_eles(lines)
        
    # fold
    lines = fold_comments(lines)
    
    # Write file in place. 
    with open(bmad_file, 'w') as f:
        for line in lines:
            f.write(line+'\n')
finalize_bmad('UND.bmad', replacements=NEWELES)            

Found ele: umasx
Found ele: umahx
Found ele: pssx
Found ele: pshx


# Convert all

In [138]:
# Clean
!rm *xsif *bmad *digested*

rm: *xsif: No such file or directory
rm: *bmad: No such file or directory
rm: *digested*: No such file or directory


In [139]:
!cp /Users/chrisonian/Code/GitHub/lcls2he-lattice/mad/*xsif .

In [140]:
XSIF_FILES=[f for f in os.listdir() if f.endswith('.xsif')]
for f in XSIF_FILES:
    prepare_xsif(f, save=False)

Preparing LCLS_L3.xsif
Preparing INJ.xsif
Preparing CU_SXR.xsif
Preparing LE_BYP.xsif
Preparing LCLS2cu_master.xsif
Preparing LCLS_L2.xsif
Preparing CU_SFTH.xsif
Preparing ALINE.xsif
Preparing DASEL.xsif
Preparing CUSXR.xsif
Preparing DIAG0.xsif
Preparing SPRD.xsif
Preparing HXTES.xsif
Preparing SC_SXRLE.xsif
Preparing DLBM.xsif
Preparing SC_SXR.xsif
Preparing CM.xsif
Preparing BC1.xsif
Preparing common.xsif
Preparing BC2.xsif
Preparing LE_SPRD.xsif
Preparing SC_BSYDLE.xsif
Preparing LCLS2cu.xsif
Preparing SFT.xsif
Preparing LCLS_L3e.xsif
Preparing LCLS2sc_master.xsif
Preparing BSYsc.xsif
Preparing SC_DASEL.xsif
Preparing UND.xsif
Preparing CU_SPEC.xsif
Preparing SC_DIAG0.xsif
Preparing CU_GSPEC.xsif
Preparing SXTES.xsif
Preparing CU_HXR.xsif
Preparing LCLS_L1e.xsif
Preparing L3EXT.xsif
Preparing LCLS_L1.xsif
Preparing BYP.xsif
Preparing SC_HXR.xsif
Preparing EXT.xsif
Preparing BSYcu.xsif
Preparing SC_SFTS.xsif
Preparing CU_ALINE.xsif
Preparing SC_BSYD.xsif
Preparing BLINE.xsif
Prepari

In [141]:
!rm *bmad
!cp temp/*xsif .

rm: *bmad: No such file or directory


In [142]:
translate_xsif_to_bmad('SC_HXR.xsif')

Translating: ['/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver', '-bmad', 'SC_HXR.xsif']
CompletedProcess(args=['/Users/chrisonian/Code/AML/accelerator-ml-code/uap/trunk/bin/translate_driver', '-bmad', 'SC_HXR.xsif'], returncode=0, stdout=b'\nOutput file: SC_HXR.bmad\n', stderr=b'')


In [104]:
#!cp save_LTU LTU.bmad

In [105]:
#finalize_bmad('LTU.bmad')

 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.
 
   Skipping.


In [145]:
BMAD_FILES=[f for f in os.listdir() if f.endswith('.bmad')]
for f in BMAD_FILES:
    finalize_bmad(f, replacements=NEWELES)   

In [144]:
!rm *xsif 

rm: *xsif: No such file or directory


In [146]:
!mv *bmad /Users/chrisonian/Code/GitHub/lcls2he-lattice/bmad/master